# 05 Strict R2G Decomposed Connectedness with Hierarchical Ward Components

This notebook implements the paper-formula R2G decomposition for precious-metal futures and spot markets using the STC/MTC/LTC reconstructed components from `02_iceemdan_hierarchical_ward_reconstruction.ipynb`.

The central formula is `R2G = R_xz^2 (R_xz^{-1} R_yx)^2`, where `R_xx = V Lambda V'`, `R_xz = V Lambda^{1/2} V'`, and the square is element-wise. Outputs mirror notebook 04 so that strict R2G risk spillovers can be compared with PCA-orthogonalized explanatory connectedness.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_ROOT = Path.cwd()
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "02_iceemdan_hierarchical_ward" / "hierarchical_ward_components_wide.csv"
SOURCE_MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "02_iceemdan_hierarchical_ward" / "run_manifest.json"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "05_strict_r2g_decomposed_connectedness_hierarchical_ward"
VIS_DIR = OUTPUT_DIR / "visualizations"

STATIC_HEATMAP_DIR = VIS_DIR / "static_heatmaps"
STATIC_NET_DIR = VIS_DIR / "static_net_barplots"
STATIC_NETWORK_DIR = VIS_DIR / "static_npdc_networks"
DYNAMIC_TCI_DIR = VIS_DIR / "dynamic_tci"
DYNAMIC_NET_DIR = VIS_DIR / "dynamic_net"
DYNAMIC_NET_HEATMAP_DIR = VIS_DIR / "dynamic_net_heatmaps"
ROBUSTNESS_TCI_DIR = VIS_DIR / "robustness_tci"

for path in [
    OUTPUT_DIR,
    VIS_DIR,
    STATIC_HEATMAP_DIR,
    STATIC_NET_DIR,
    STATIC_NETWORK_DIR,
    DYNAMIC_TCI_DIR,
    DYNAMIC_NET_DIR,
    DYNAMIC_NET_HEATMAP_DIR,
    ROBUSTNESS_TCI_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

METAL_ORDER = ["gold", "silver", "platinum", "palladium"]
MARKET_ORDER = ["futures", "spot"]
SCALE_ORDER = ["STC", "MTC", "LTC"]
RETURN_SERIES_ORDER = [
    "gold_futures",
    "silver_futures",
    "platinum_futures",
    "palladium_futures",
    "gold_spot",
    "silver_spot",
    "platinum_spot",
    "palladium_spot",
]
EXPECTED_COLUMNS = ["date", *[f"{series}_{scale}" for series in RETURN_SERIES_ORDER for scale in SCALE_ORDER]]

PARAMS = {
    "method": "Strict paper-formula R2G decomposed connectedness on standardized reconstructed return components",
    "main_window_size": 252,
    "robustness_window_sizes": [200, 300],
    "r2_tolerance": 1e-8,
    "eigenvalue_tolerance": 1e-12,
    "percent_scale": True,
    "static_full_sample": True,
    "dynamic_detail_window_sizes": [252],
}

plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

start_time = time.perf_counter()
print(f"Input: {INPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Output: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")


Input: data\processed\02_iceemdan_hierarchical_ward\hierarchical_ward_components_wide.csv
Output: data\processed\05_strict_r2g_decomposed_connectedness_hierarchical_ward


## Input Checks

The notebook uses the Hierarchical Ward reconstructed component table as the sole input for connectedness estimation.

In [2]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


components_wide = pd.read_csv(INPUT_PATH, parse_dates=["date"])

assert components_wide.shape == (3100, 25), components_wide.shape
assert list(components_wide.columns) == EXPECTED_COLUMNS, components_wide.columns.tolist()
assert components_wide["date"].min() == pd.Timestamp("2012-01-04"), components_wide["date"].min()
assert components_wide["date"].max() == pd.Timestamp("2026-03-31"), components_wide["date"].max()
assert components_wide["date"].is_monotonic_increasing
assert components_wide["date"].duplicated().sum() == 0
assert components_wide.drop(columns=["date"]).isna().sum().sum() == 0
assert np.isfinite(components_wide.drop(columns=["date"]).to_numpy()).all()
assert all(components_wide[col].var() > 0 for col in EXPECTED_COLUMNS if col != "date")

input_sha256 = sha256_file(INPUT_PATH)
source_manifest_sha256 = sha256_file(SOURCE_MANIFEST_PATH) if SOURCE_MANIFEST_PATH.exists() else None
source_manifest = json.loads(SOURCE_MANIFEST_PATH.read_text(encoding="utf-8")) if SOURCE_MANIFEST_PATH.exists() else {}

input_checks = {
    "input_file": str(INPUT_PATH.relative_to(PROJECT_ROOT)),
    "input_sha256": input_sha256,
    "source_manifest_file": str(SOURCE_MANIFEST_PATH.relative_to(PROJECT_ROOT)) if SOURCE_MANIFEST_PATH.exists() else None,
    "source_manifest_sha256": source_manifest_sha256,
    "n_rows": int(len(components_wide)),
    "n_component_columns": int(len(components_wide.columns) - 1),
    "date_start": components_wide["date"].min().strftime("%Y-%m-%d"),
    "date_end": components_wide["date"].max().strftime("%Y-%m-%d"),
    "duplicate_dates": int(components_wide["date"].duplicated().sum()),
    "missing_values": int(components_wide.drop(columns=["date"]).isna().sum().sum()),
}

input_checks

{'input_file': 'data\\processed\\02_iceemdan_hierarchical_ward\\hierarchical_ward_components_wide.csv',
 'input_sha256': '6519950771bcc330efadda9ed6ae375cb7b9c455cf8443cd4af984234e460480',
 'source_manifest_file': 'data\\processed\\02_iceemdan_hierarchical_ward\\run_manifest.json',
 'source_manifest_sha256': 'a29ff9191f9ddbd9f6d53a9a5d189f0b563d5d32d5682390bb9e9adda4ebb1f1',
 'n_rows': 3100,
 'n_component_columns': 24,
 'date_start': '2012-01-04',
 'date_end': '2026-03-31',
 'duplicate_dates': 0,
 'missing_values': 0}

## Network Definitions

Two families of connectedness networks are constructed: same-scale cross-metal networks and within-metal cross-scale networks.

In [3]:
@dataclass(frozen=True)
class NetworkDefinition:
    network_id: str
    network_type: str
    node_columns: tuple[str, ...]
    scale: str | None = None
    metal: str | None = None


def parse_component_column(column: str) -> dict[str, str]:
    series, scale = column.rsplit("_", 1)
    metal, market = series.rsplit("_", 1)
    return {"series": series, "metal": metal, "market": market, "scale": scale}


METAL_ABBREVIATIONS = {"gold": "AU", "silver": "AG", "platinum": "PT", "palladium": "PD"}
MARKET_ABBREVIATIONS = {"futures": "F", "spot": "S"}


def display_label(column: str, network: NetworkDefinition) -> str:
    meta = parse_component_column(column)
    metal = METAL_ABBREVIATIONS[meta["metal"]]
    market = MARKET_ABBREVIATIONS[meta["market"]]
    scale = meta["scale"]
    if network.network_type == "same_scale_cross_metal":
        return f"{metal}_{market}"
    return f"{market}_{scale}"


networks: list[NetworkDefinition] = []

for scale in SCALE_ORDER:
    networks.append(
        NetworkDefinition(
            network_id=f"same_scale_{scale}",
            network_type="same_scale_cross_metal",
            scale=scale,
            node_columns=tuple(f"{series}_{scale}" for series in RETURN_SERIES_ORDER),
        )
    )

for metal in METAL_ORDER:
    node_columns = []
    for scale in SCALE_ORDER:
        for market in MARKET_ORDER:
            node_columns.append(f"{metal}_{market}_{scale}")
    networks.append(
        NetworkDefinition(
            network_id=f"within_{metal}",
            network_type="within_metal_cross_scale",
            metal=metal,
            node_columns=tuple(node_columns),
        )
    )

network_metadata_rows = []
for network in networks:
    for order, column in enumerate(network.node_columns, start=1):
        meta = parse_component_column(column)
        network_metadata_rows.append(
            {
                "network_id": network.network_id,
                "network_type": network.network_type,
                "network_scale": network.scale,
                "network_metal": network.metal,
                "node_order": order,
                "node_id": column,
                "display_label": display_label(column, network),
                "component_column": column,
                **meta,
            }
        )

network_node_metadata = pd.DataFrame(network_metadata_rows)
network_node_metadata.to_csv(OUTPUT_DIR / "network_node_metadata.csv", index=False)

assert len(networks) == 7
assert set(network_node_metadata["network_id"]) == {network.network_id for network in networks}
assert network_node_metadata.groupby("network_id").size().to_dict() == {
    "same_scale_STC": 8,
    "same_scale_MTC": 8,
    "same_scale_LTC": 8,
    "within_gold": 6,
    "within_silver": 6,
    "within_platinum": 6,
    "within_palladium": 6,
}

network_node_metadata.head(12)


,network_id,network_type,network_scale,network_metal,node_order,node_id,display_label,component_column,series,metal,market,scale
0,same_scale_STC,same_scale_cross_metal,STC,NaN,1,gold_futures_STC,Gold Futures,gold_futures_STC,gold_futures,gold,futures,STC
1,same_scale_STC,same_scale_cross_metal,STC,NaN,2,silver_futures_STC,Silver Futures,silver_futures_STC,silver_futures,silver,futures,STC
2,same_scale_STC,same_scale_cross_metal,STC,NaN,3,platinum_futures_STC,Platinum Futures,platinum_futures_STC,platinum_futures,platinum,futures,STC
3,same_scale_STC,same_scale_cross_metal,STC,NaN,4,palladium_futures_STC,Palladium Futures,palladium_futures_STC,palladium_futures,palladium,futures,STC
4,same_scale_STC,same_scale_cross_metal,STC,NaN,5,gold_spot_STC,Gold Spot,gold_spot_STC,gold_spot,gold,spot,STC
5,same_scale_STC,same_scale_cross_metal,STC,NaN,6,silver_spot_STC,Silver Spot,silver_spot_STC,silver_spot,silver,spot,STC
6,same_scale_STC,same_scale_cross_metal,STC,NaN,7,platinum_spot_STC,Platinum Spot,platinum_spot_STC,platinum_spot,platinum,spot,STC
7,same_scale_STC,same_scale_cross_metal,STC,NaN,8,palladium_spot_STC,Palladium Spot,palladium_spot_STC,palladium_spot,palladium,spot,STC
8,same_scale_MTC,same_scale_cross_metal,MTC,NaN,1,gold_futures_MTC,Gold Futures,gold_futures_MTC,gold_futures,gold,futures,MTC
9,same_scale_MTC,same_scale_cross_metal,MTC,NaN,2,silver_futures_MTC,Silver Futures,silver_futures_MTC,silver_futures,silver,futures,MTC


## Strict Paper-Formula R2G Helpers

For each target node, this notebook decomposes the multiple-regression `R2` by the paper's strict `R2G` formula. Let `R_xx = V Lambda V'` and `R_xz = V Lambda^{1/2} V'`. Contributions are computed as `R_xz^2 (R_xz^{-1} R_yx)^2`, with element-wise squares. For each target node, the source contributions sum to the full multiple-regression `R2` up to numerical tolerance.

In [4]:
def standardized_correlation(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if arr.ndim != 2:
        raise ValueError("values must be a 2D array")
    if arr.shape[0] <= arr.shape[1]:
        raise ValueError(f"Need more observations than variables, got {arr.shape}")
    means = arr.mean(axis=0)
    stds = arr.std(axis=0, ddof=0)
    if np.any(stds <= eps):
        bad = np.where(stds <= eps)[0].tolist()
        raise ValueError(f"Near-constant columns at positions {bad}")
    z = (arr - means) / stds
    corr = (z.T @ z) / z.shape[0]
    corr = (corr + corr.T) / 2.0
    corr = np.clip(corr, -1.0, 1.0)
    np.fill_diagonal(corr, 1.0)
    return corr


def nearest_psd(matrix: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=float)
    matrix = (matrix + matrix.T) / 2.0
    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.maximum(eigvals, eps)
    repaired = (eigvecs * eigvals) @ eigvecs.T
    repaired = (repaired + repaired.T) / 2.0
    diag = np.sqrt(np.maximum(np.diag(repaired), eps))
    repaired = repaired / np.outer(diag, diag)
    repaired = (repaired + repaired.T) / 2.0
    np.fill_diagonal(repaired, 1.0)
    return repaired


def strict_r2g_contributions_from_corr(
    predictor_corr: np.ndarray,
    target_corr: np.ndarray,
    eigenvalue_tol: float = 1e-12,
) -> tuple[np.ndarray, float]:
    predictor_corr = nearest_psd(predictor_corr, eps=eigenvalue_tol)
    target_corr = np.asarray(target_corr, dtype=float)

    eigvals, eigvecs = np.linalg.eigh(predictor_corr)
    eigvals = np.maximum(eigvals, eigenvalue_tol)
    rxz = (eigvecs * np.sqrt(eigvals)) @ eigvecs.T
    beta = np.linalg.solve(rxz, target_corr)
    rxz_sq = rxz**2
    beta_sq = beta**2
    contributions = rxz_sq @ beta_sq
    contributions = np.clip(contributions, 0.0, None)

    total_r2 = float(target_corr @ np.linalg.solve(predictor_corr, target_corr))
    total_r2 = float(np.clip(total_r2, 0.0, 1.0))
    contribution_sum = float(contributions.sum())
    if contribution_sum > eigenvalue_tol:
        contributions *= total_r2 / contribution_sum
    else:
        contributions = np.zeros_like(contributions)
    contributions = np.clip(contributions, 0.0, None)
    contribution_sum = float(contributions.sum())
    if contribution_sum > eigenvalue_tol:
        contributions *= total_r2 / contribution_sum
    return contributions, total_r2


def r2g_matrix(
    values: np.ndarray,
    node_ids: list[str],
    eigenvalue_tol: float = 1e-12,
) -> tuple[np.ndarray, pd.DataFrame]:
    corr = standardized_correlation(values, eps=eigenvalue_tol)
    k = len(node_ids)
    matrix = np.zeros((k, k), dtype=float)
    validation_rows = []

    for target_idx, target_node in enumerate(node_ids):
        predictor_idx = [idx for idx in range(k) if idx != target_idx]
        predictor_corr = corr[np.ix_(predictor_idx, predictor_idx)]
        target_corr = corr[predictor_idx, target_idx]
        contributions, total_r2 = strict_r2g_contributions_from_corr(
            predictor_corr,
            target_corr,
            eigenvalue_tol=eigenvalue_tol,
        )
        matrix[target_idx, predictor_idx] = contributions
        validation_rows.append(
            {
                "target": target_node,
                "total_r2": total_r2,
                "contribution_sum": float(contributions.sum()),
                "row_error": float(abs(contributions.sum() - total_r2)),
                "min_contribution": float(contributions.min()),
                "max_contribution": float(contributions.max()),
            }
        )

    return matrix, pd.DataFrame(validation_rows)


def matrix_to_long(
    matrix: np.ndarray,
    network: NetworkDefinition,
    date: pd.Timestamp | None = None,
    window_size: int | None = None,
    window_start: pd.Timestamp | None = None,
    window_end: pd.Timestamp | None = None,
) -> list[dict[str, Any]]:
    rows = []
    for target_idx, target in enumerate(network.node_columns):
        for source_idx, source in enumerate(network.node_columns):
            if target_idx == source_idx:
                continue
            value = float(matrix[target_idx, source_idx])
            rows.append(
                {
                    "date": date,
                    "window_size": window_size,
                    "window_start": window_start,
                    "window_end": window_end,
                    "network_id": network.network_id,
                    "network_type": network.network_type,
                    "network_scale": network.scale,
                    "network_metal": network.metal,
                    "target": target,
                    "source": source,
                    "r2g": value,
                    "r2g_percent": value * 100.0,
                }
            )
    return rows


def directional_rows(
    matrix: np.ndarray,
    network: NetworkDefinition,
    date: pd.Timestamp | None = None,
    window_size: int | None = None,
    window_start: pd.Timestamp | None = None,
    window_end: pd.Timestamp | None = None,
) -> list[dict[str, Any]]:
    rows = []
    from_values = matrix.sum(axis=1) * 100.0
    to_values = matrix.sum(axis=0) * 100.0
    for idx, node in enumerate(network.node_columns):
        rows.append(
            {
                "date": date,
                "window_size": window_size,
                "window_start": window_start,
                "window_end": window_end,
                "network_id": network.network_id,
                "network_type": network.network_type,
                "network_scale": network.scale,
                "network_metal": network.metal,
                "node": node,
                "FROM": float(from_values[idx]),
                "TO": float(to_values[idx]),
                "NET": float(to_values[idx] - from_values[idx]),
            }
        )
    return rows


def npdc_rows(
    matrix: np.ndarray,
    network: NetworkDefinition,
    date: pd.Timestamp | None = None,
    window_size: int | None = None,
    window_start: pd.Timestamp | None = None,
    window_end: pd.Timestamp | None = None,
) -> list[dict[str, Any]]:
    rows = []
    for source_idx, source in enumerate(network.node_columns):
        for target_idx, target in enumerate(network.node_columns):
            if source_idx == target_idx:
                continue
            source_to_target = matrix[target_idx, source_idx] * 100.0
            target_to_source = matrix[source_idx, target_idx] * 100.0
            rows.append(
                {
                    "date": date,
                    "window_size": window_size,
                    "window_start": window_start,
                    "window_end": window_end,
                    "network_id": network.network_id,
                    "network_type": network.network_type,
                    "network_scale": network.scale,
                    "network_metal": network.metal,
                    "source": source,
                    "target": target,
                    "source_to_target": float(source_to_target),
                    "target_to_source": float(target_to_source),
                    "NPDC": float(source_to_target - target_to_source),
                }
            )
    return rows


def tci_row(
    matrix: np.ndarray,
    network: NetworkDefinition,
    date: pd.Timestamp | None = None,
    window_size: int | None = None,
    window_start: pd.Timestamp | None = None,
    window_end: pd.Timestamp | None = None,
) -> dict[str, Any]:
    row_sums = matrix.sum(axis=1)
    return {
        "date": date,
        "window_size": window_size,
        "window_start": window_start,
        "window_end": window_end,
        "network_id": network.network_id,
        "network_type": network.network_type,
        "network_scale": network.scale,
        "network_metal": network.metal,
        "TCI": float(row_sums.mean() * 100.0),
        "mean_total_r2": float(row_sums.mean()),
        "min_total_r2": float(row_sums.min()),
        "max_total_r2": float(row_sums.max()),
    }


## Static Full-Sample Connectedness

Static explanatory connectedness is estimated on the full reconstructed-component sample.

In [5]:
static_matrix_rows: list[dict[str, Any]] = []
static_directional_rows: list[dict[str, Any]] = []
static_npdc_rows: list[dict[str, Any]] = []
static_tci_rows: list[dict[str, Any]] = []
static_validation_rows: list[dict[str, Any]] = []
static_matrices: dict[str, np.ndarray] = {}

full_start = components_wide["date"].iloc[0]
full_end = components_wide["date"].iloc[-1]

for network in networks:
    values = components_wide.loc[:, list(network.node_columns)].to_numpy(dtype=float)
    matrix, validation = r2g_matrix(
        values,
        list(network.node_columns),
        eigenvalue_tol=PARAMS["eigenvalue_tolerance"],
    )
    static_matrices[network.network_id] = matrix
    validation.insert(0, "network_id", network.network_id)
    validation.insert(1, "window_size", len(components_wide))
    validation.insert(2, "window_start", full_start)
    validation.insert(3, "window_end", full_end)
    static_validation_rows.extend(validation.to_dict("records"))

    static_matrix_rows.extend(
        matrix_to_long(matrix, network, window_size=len(components_wide), window_start=full_start, window_end=full_end)
    )
    static_directional_rows.extend(
        directional_rows(matrix, network, window_size=len(components_wide), window_start=full_start, window_end=full_end)
    )
    static_npdc_rows.extend(
        npdc_rows(matrix, network, window_size=len(components_wide), window_start=full_start, window_end=full_end)
    )
    static_tci_rows.append(
        tci_row(matrix, network, window_size=len(components_wide), window_start=full_start, window_end=full_end)
    )

static_r2g_matrices_long = pd.DataFrame(static_matrix_rows)
static_directional_indices = pd.DataFrame(static_directional_rows)
static_npdc = pd.DataFrame(static_npdc_rows)
static_tci = pd.DataFrame(static_tci_rows)
static_validation = pd.DataFrame(static_validation_rows)

assert static_validation["row_error"].max() < PARAMS["r2_tolerance"]
assert static_r2g_matrices_long["r2g"].min() >= -PARAMS["r2_tolerance"]

static_r2g_matrices_long.to_csv(OUTPUT_DIR / "static_r2g_matrices_long.csv", index=False)
static_directional_indices.to_csv(OUTPUT_DIR / "static_directional_indices.csv", index=False)
static_npdc.to_csv(OUTPUT_DIR / "static_npdc.csv", index=False)
static_tci.to_csv(OUTPUT_DIR / "static_tci.csv", index=False)
static_validation.to_csv(OUTPUT_DIR / "static_r2_decomposition_validation.csv", index=False)

static_tci

,date,window_size,window_start,window_end,network_id,network_type,network_scale,network_metal,TCI,mean_total_r2,min_total_r2,max_total_r2
0,None,3100,2012-01-04,2026-03-31,same_scale_STC,same_scale_cross_metal,STC,NaN,79.510360,0.795104,0.622962,0.885223
1,None,3100,2012-01-04,2026-03-31,same_scale_MTC,same_scale_cross_metal,MTC,NaN,65.334276,0.653343,0.115359,0.877486
2,None,3100,2012-01-04,2026-03-31,same_scale_LTC,same_scale_cross_metal,LTC,NaN,70.787508,0.707875,0.591603,0.823226
3,None,3100,2012-01-04,2026-03-31,within_gold,within_metal_cross_scale,NaN,gold,57.576710,0.575767,0.427603,0.703990
4,None,3100,2012-01-04,2026-03-31,within_silver,within_metal_cross_scale,NaN,silver,49.404760,0.494048,0.195397,0.750371
5,None,3100,2012-01-04,2026-03-31,within_platinum,within_metal_cross_scale,NaN,platinum,84.686290,0.846863,0.798309,0.891780
6,None,3100,2012-01-04,2026-03-31,within_palladium,within_metal_cross_scale,NaN,palladium,77.467253,0.774673,0.582302,0.881196


## Rolling Connectedness

The main dynamic explanatory-connectedness results use a 252-trading-day rolling window. Robustness checks use 200-day and 300-day windows for TCI.

In [6]:
def compute_rolling_connectedness(
    network: NetworkDefinition,
    window_size: int,
    include_directional: bool = True,
    include_npdc: bool = True,
) -> dict[str, pd.DataFrame]:
    dates = components_wide["date"].reset_index(drop=True)
    values = components_wide.loc[:, list(network.node_columns)].to_numpy(dtype=float)
    n_obs = values.shape[0]

    tci_rows_out: list[dict[str, Any]] = []
    directional_rows_out: list[dict[str, Any]] = []
    npdc_rows_out: list[dict[str, Any]] = []
    validation_rows_out: list[dict[str, Any]] = []

    for end_idx in range(window_size - 1, n_obs):
        start_idx = end_idx - window_size + 1
        sample = values[start_idx : end_idx + 1]
        window_start = dates.iloc[start_idx]
        window_end = dates.iloc[end_idx]
        matrix, validation = r2g_matrix(
            sample,
            list(network.node_columns),
            eigenvalue_tol=PARAMS["eigenvalue_tolerance"],
        )

        tci_rows_out.append(tci_row(matrix, network, date=window_end, window_size=window_size, window_start=window_start, window_end=window_end))

        if include_directional:
            directional_rows_out.extend(
                directional_rows(matrix, network, date=window_end, window_size=window_size, window_start=window_start, window_end=window_end)
            )
        if include_npdc:
            npdc_rows_out.extend(
                npdc_rows(matrix, network, date=window_end, window_size=window_size, window_start=window_start, window_end=window_end)
            )

        validation_rows_out.append(
            {
                "date": window_end,
                "network_id": network.network_id,
                "window_size": window_size,
                "window_start": window_start,
                "window_end": window_end,
                "max_row_error": float(validation["row_error"].max()),
                "min_contribution": float(validation["min_contribution"].min()),
                "max_contribution": float(validation["max_contribution"].max()),
            }
        )

    return {
        "tci": pd.DataFrame(tci_rows_out),
        "directional": pd.DataFrame(directional_rows_out),
        "npdc": pd.DataFrame(npdc_rows_out),
        "validation": pd.DataFrame(validation_rows_out),
    }


main_window = PARAMS["main_window_size"]
main_tci_frames = []
main_directional_frames = []
main_npdc_frames = []
main_validation_frames = []
rolling_start_time = time.perf_counter()

for network in networks:
    network_start = time.perf_counter()
    result = compute_rolling_connectedness(network, main_window, include_directional=True, include_npdc=True)
    main_tci_frames.append(result["tci"])
    main_directional_frames.append(result["directional"])
    main_npdc_frames.append(result["npdc"])
    main_validation_frames.append(result["validation"])
    print(f"{network.network_id}: {len(result['tci'])} rolling windows in {time.perf_counter() - network_start:.1f}s")

rolling_elapsed = time.perf_counter() - rolling_start_time

dynamic_tci_window252 = pd.concat(main_tci_frames, ignore_index=True)
dynamic_directional_indices_window252 = pd.concat(main_directional_frames, ignore_index=True)
dynamic_npdc_window252 = pd.concat(main_npdc_frames, ignore_index=True)
dynamic_validation_window252 = pd.concat(main_validation_frames, ignore_index=True)

expected_main_tci_rows = (len(components_wide) - main_window + 1) * len(networks)
assert len(dynamic_tci_window252) == expected_main_tci_rows
assert dynamic_validation_window252["max_row_error"].max() < PARAMS["r2_tolerance"]
assert np.isfinite(dynamic_tci_window252["TCI"]).all()

for df in [dynamic_tci_window252, dynamic_directional_indices_window252, dynamic_npdc_window252, dynamic_validation_window252]:
    for col in ["date", "window_start", "window_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d")

dynamic_tci_window252.to_csv(OUTPUT_DIR / "dynamic_tci_window252.csv", index=False)
dynamic_directional_indices_window252.to_csv(OUTPUT_DIR / "dynamic_directional_indices_window252.csv", index=False)
dynamic_npdc_window252.to_csv(OUTPUT_DIR / "dynamic_npdc_window252.csv", index=False)
dynamic_validation_window252.to_csv(OUTPUT_DIR / "dynamic_r2_decomposition_validation_window252.csv", index=False)

print(f"Main rolling connectedness finished in {rolling_elapsed:.1f}s")
dynamic_tci_window252.head()

same_scale_STC: 2849 rolling windows in 4.4s


same_scale_MTC: 2849 rolling windows in 4.6s


same_scale_LTC: 2849 rolling windows in 4.6s


within_gold: 2849 rolling windows in 3.6s


within_silver: 2849 rolling windows in 3.6s


within_platinum: 2849 rolling windows in 3.6s


within_palladium: 2849 rolling windows in 3.7s


Main rolling connectedness finished in 28.1s


,date,window_size,window_start,window_end,network_id,network_type,network_scale,network_metal,TCI,mean_total_r2,min_total_r2,max_total_r2
0,2013-01-02,252,2012-01-04,2013-01-02,same_scale_STC,same_scale_cross_metal,STC,None,88.432814,0.884328,0.776410,0.948322
1,2013-01-03,252,2012-01-05,2013-01-03,same_scale_STC,same_scale_cross_metal,STC,None,88.368636,0.883686,0.777705,0.947615
2,2013-01-04,252,2012-01-06,2013-01-04,same_scale_STC,same_scale_cross_metal,STC,None,88.299810,0.882998,0.778988,0.946095
3,2013-01-07,252,2012-01-09,2013-01-07,same_scale_STC,same_scale_cross_metal,STC,None,88.210435,0.882104,0.778626,0.945218
4,2013-01-08,252,2012-01-10,2013-01-08,same_scale_STC,same_scale_cross_metal,STC,None,88.222037,0.882220,0.779460,0.945209


In [7]:
robustness_tci_frames = []
robustness_validation_frames = []

for window_size in PARAMS["robustness_window_sizes"]:
    for network in networks:
        network_start = time.perf_counter()
        result = compute_rolling_connectedness(network, window_size, include_directional=False, include_npdc=False)
        robustness_tci_frames.append(result["tci"])
        robustness_validation_frames.append(result["validation"])
        print(f"{network.network_id}, window={window_size}: {len(result['tci'])} TCI rows in {time.perf_counter() - network_start:.1f}s")

robustness_tci = pd.concat(robustness_tci_frames, ignore_index=True)
robustness_validation = pd.concat(robustness_validation_frames, ignore_index=True)

expected_robust_rows = sum((len(components_wide) - window + 1) * len(networks) for window in PARAMS["robustness_window_sizes"])
assert len(robustness_tci) == expected_robust_rows
assert robustness_validation["max_row_error"].max() < PARAMS["r2_tolerance"]

for df in [robustness_tci, robustness_validation]:
    for col in ["date", "window_start", "window_end"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d")

robustness_tci.to_csv(OUTPUT_DIR / "dynamic_tci_robustness_windows_200_300.csv", index=False)
robustness_validation.to_csv(OUTPUT_DIR / "dynamic_r2_decomposition_validation_robustness_windows_200_300.csv", index=False)

robustness_tci.groupby("window_size").size()

same_scale_STC, window=200: 2901 TCI rows in 3.9s


same_scale_MTC, window=200: 2901 TCI rows in 4.1s


same_scale_LTC, window=200: 2901 TCI rows in 3.9s


within_gold, window=200: 2901 TCI rows in 3.3s


within_silver, window=200: 2901 TCI rows in 3.8s


...

within_gold, window=300: 2801 TCI rows in 3.2s


within_silver, window=300: 2801 TCI rows in 3.3s


within_platinum, window=300: 2801 TCI rows in 3.3s


within_palladium, window=300: 2801 TCI rows in 3.3s


window_size
200    20307
300    19607
dtype: int64

## Visualizations

The visualizations mirror the data tables: static strict R2G heatmaps, static NET bars, static NPDC networks, dynamic TCI, dynamic NET, and robustness TCI comparisons.

In [8]:
def labels_for_network(network: NetworkDefinition) -> list[str]:
    return [display_label(column, network) for column in network.node_columns]


def save_figure(fig: plt.Figure, directory: Path, filename_stem: str) -> None:
    fig.tight_layout()
    fig.savefig(directory / f"{filename_stem}.png", bbox_inches="tight")
    fig.savefig(directory / f"{filename_stem}.pdf", bbox_inches="tight")
    plt.close(fig)


def heatmap_colorbar_label() -> str:
    method = str(PARAMS.get("method", "")).lower()
    if "pca" in method:
        return "PCA R2 contribution (%)"
    return "R2G contribution (%)"


def plot_static_heatmap(network: NetworkDefinition, matrix: np.ndarray) -> None:
    labels = labels_for_network(network)
    fig, ax = plt.subplots(figsize=(8.2, 6.4))
    im = ax.imshow(matrix * 100.0, cmap="YlOrRd", aspect="auto")
    ax.set_title(f"R2G contribution matrix: {network.network_id}")
    ax.set_xlabel("Source")
    ax.set_ylabel("Target")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    max_value = np.nanmax(matrix * 100.0)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j] * 100.0
            text_color = "white" if value > max_value * 0.55 else "black"
            ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=7, color=text_color)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(heatmap_colorbar_label())
    save_figure(fig, STATIC_HEATMAP_DIR, f"{network.network_id}_static_r2g_heatmap")


def _format_network_title(network_id: str) -> str:
    if network_id.startswith("same_scale_"):
        return network_id.replace("same_scale_", "").upper()
    if network_id.startswith("within_"):
        return network_id.replace("within_", "").replace("_", " ").title()
    return network_id.replace("_", " ").title()


def plot_static_heatmap_group(network_ids: list[str], filename_stem: str, title: str) -> None:
    n = len(network_ids)
    ncols = 3 if n == 3 else 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols + 0.9, 4.0 * nrows), squeeze=False)
    vmax = max(float(np.nanmax(static_matrices[network_id] * 100.0)) for network_id in network_ids)
    last_im = None
    for ax, network_id in zip(axes.ravel(), network_ids):
        network = next(item for item in networks if item.network_id == network_id)
        matrix = static_matrices[network_id] * 100.0
        labels = labels_for_network(network)
        last_im = ax.imshow(matrix, cmap="YlOrRd", aspect="auto", vmin=0.0, vmax=vmax)
        ax.set_title(_format_network_title(network_id), fontsize=12, pad=8)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=9)
        for i in range(matrix.shape[0]):
            for j in range(matrix.shape[1]):
                value = matrix[i, j]
                text_color = "white" if value > vmax * 0.58 else "black"
                ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=6.2, color=text_color)
    for ax in axes.ravel()[n:]:
        ax.axis("off")
    fig.suptitle(title, fontsize=13, y=0.98)
    if nrows == 1:
        fig.subplots_adjust(left=0.06, right=0.88, bottom=0.23, top=0.78, wspace=0.30)
        cax = fig.add_axes([0.905, 0.24, 0.018, 0.50])
    else:
        fig.subplots_adjust(left=0.08, right=0.88, bottom=0.13, top=0.90, wspace=0.28, hspace=0.48)
        cax = fig.add_axes([0.905, 0.16, 0.018, 0.68])
    cbar = fig.colorbar(last_im, cax=cax)
    cbar.set_label(heatmap_colorbar_label())
    fig.savefig(STATIC_HEATMAP_DIR / f"{filename_stem}.png", bbox_inches="tight")
    fig.savefig(STATIC_HEATMAP_DIR / f"{filename_stem}.pdf", bbox_inches="tight")
    plt.close(fig)


def plot_static_net_bar(network: NetworkDefinition, directional: pd.DataFrame) -> None:
    data = directional.loc[directional["network_id"].eq(network.network_id)].copy()
    label_map = {column: display_label(column, network) for column in network.node_columns}
    data["label"] = data["node"].map(label_map)
    data = data.sort_values("NET", ascending=True)
    colors = np.where(data["NET"].to_numpy() >= 0, "#2a9d8f", "#d1495b")
    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    ax.barh(data["label"], data["NET"], color=colors)
    ax.axvline(0, color="#222222", linewidth=0.8)
    ax.set_title(f"Net strict R2G connectedness: {network.network_id}")
    ax.set_xlabel("NET (%)")
    save_figure(fig, STATIC_NET_DIR, f"{network.network_id}_static_net_barplot")


def _network_label_position(point: np.ndarray, center: np.ndarray) -> tuple[float, float, str, str]:
    """Place labels near, not on top of, circular-layout nodes."""
    point = np.asarray(point, dtype=float)
    direction = point - center
    norm = float(np.linalg.norm(direction))
    if norm < 1e-12:
        direction = np.array([1.0, 0.0])
    else:
        direction = direction / norm
    tangent = np.array([-direction[1], direction[0]])
    label_point = point + 0.17 * direction + 0.12 * tangent
    if abs(direction[0]) <= 0.25:
        ha = "right" if tangent[0] < 0 else "left"
    else:
        ha = "left" if direction[0] > 0 else "right"
    va = "bottom" if direction[1] > 0.25 else "top" if direction[1] < -0.25 else "center"
    return float(label_point[0]), float(label_point[1]), ha, va


def plot_static_npdc_network(network: NetworkDefinition, npdc_df: pd.DataFrame) -> None:
    from matplotlib.patches import Circle, FancyArrowPatch

    data = npdc_df.loc[npdc_df["network_id"].eq(network.network_id) & npdc_df["NPDC"].gt(0)].copy()
    data = data.sort_values("NPDC", ascending=False).head(min(14, len(data)))
    label_map = {column: display_label(column, network) for column in network.node_columns}

    graph = nx.DiGraph()
    for column in network.node_columns:
        graph.add_node(column, label=label_map[column])
    for row in data.itertuples(index=False):
        graph.add_edge(row.source, row.target, weight=float(row.NPDC))

    fig, ax = plt.subplots(figsize=(8.2, 6.4))
    pos = nx.circular_layout(graph)
    center = np.mean(np.vstack([pos[node] for node in graph.nodes]), axis=0)
    node_radius = 0.115

    node_patches: dict[str, Circle] = {}
    for node in graph.nodes:
        node_color = "#4c78a8" if parse_component_column(node)["market"] == "futures" else "#f58518"
        patch = Circle(
            xy=pos[node],
            radius=node_radius,
            facecolor=node_color,
            edgecolor="white",
            linewidth=1.25,
            zorder=3,
        )
        ax.add_patch(patch)
        node_patches[node] = patch

    if graph.number_of_edges() > 0:
        edges = list(graph.edges)
        weights = np.array([graph[u][v]["weight"] for u, v in edges], dtype=float)
        widths = 0.9 + 3.4 * (weights / weights.max())
        for (u, v), width in zip(edges, widths):
            arrow = FancyArrowPatch(
                posA=pos[u],
                posB=pos[v],
                patchA=node_patches[u],
                patchB=node_patches[v],
                shrinkA=2.0,
                shrinkB=5.5,
                arrowstyle="-|>",
                mutation_scale=20,
                linewidth=float(width),
                color="#555555",
                alpha=0.92,
                connectionstyle="arc3,rad=0.12",
                zorder=4,
            )
            ax.add_patch(arrow)

    for node in graph.nodes:
        x, y, ha, va = _network_label_position(pos[node], center)
        ax.text(
            x,
            y,
            label_map[node],
            ha=ha,
            va=va,
            fontsize=20,
            color="#222222",
            zorder=5,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.78, "pad": 0.12},
        )

    ax.set_title(_format_network_title(network.network_id), fontsize=12, pad=6)
    ax.set_aspect("equal")
    ax.set_xlim(-1.45, 1.45)
    ax.set_ylim(-1.35, 1.35)
    ax.axis("off")
    save_figure(fig, STATIC_NETWORK_DIR, f"{network.network_id}_static_npdc_network")


for network in networks:
    plot_static_heatmap(network, static_matrices[network.network_id])
    plot_static_net_bar(network, static_directional_indices)
    plot_static_npdc_network(network, static_npdc)

same_scale_ids = [f"same_scale_{scale}" for scale in SCALE_ORDER]
within_ids = [f"within_{metal}" for metal in METAL_ORDER]
plot_static_heatmap_group(same_scale_ids, "same_scale_static_r2g_heatmap_group", "Static R2G contribution heatmaps: same-scale networks")
plot_static_heatmap_group(within_ids, "within_metal_static_r2g_heatmap_group", "Static R2G contribution heatmaps: within-metal networks")

print("Static visualizations saved.")


Static visualizations saved.


In [9]:
def plot_tci_group(df: pd.DataFrame, network_ids: list[str], title: str, filename_stem: str, directory: Path = DYNAMIC_TCI_DIR) -> None:
    plot_df = df.loc[df["network_id"].isin(network_ids)].copy()
    plot_df["date"] = pd.to_datetime(plot_df["date"])
    fig, ax = plt.subplots(figsize=(10.8, 5.2))
    for network_id, group in plot_df.groupby("network_id", sort=False):
        group = group.sort_values("date")
        ax.plot(group["date"], group["TCI"], linewidth=1.2, label=network_id)
    ax.set_title(title)
    ax.set_ylabel("TCI (%)")
    ax.set_xlabel("Window end date")
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend(ncol=2, fontsize=8, frameon=False)
    save_figure(fig, directory, filename_stem)


same_scale_ids = [f"same_scale_{scale}" for scale in SCALE_ORDER]
within_ids = [f"within_{metal}" for metal in METAL_ORDER]

plot_tci_group(dynamic_tci_window252, [network.network_id for network in networks], "Dynamic TCI, 252-day rolling window", "all_networks_dynamic_tci_window252")
plot_tci_group(dynamic_tci_window252, same_scale_ids, "Same-scale dynamic TCI, 252-day rolling window", "same_scale_dynamic_tci_window252")
plot_tci_group(dynamic_tci_window252, within_ids, "Within-metal dynamic TCI, 252-day rolling window", "within_metal_dynamic_tci_window252")

for network in networks:
    data = dynamic_directional_indices_window252.loc[dynamic_directional_indices_window252["network_id"].eq(network.network_id)].copy()
    data["date"] = pd.to_datetime(data["date"])
    label_map = {column: display_label(column, network) for column in network.node_columns}
    fig, ax = plt.subplots(figsize=(10.8, 5.2))
    for node, group in data.groupby("node", sort=False):
        group = group.sort_values("date")
        ax.plot(group["date"], group["NET"], linewidth=1.0, label=label_map[node])
    ax.axhline(0, color="#222222", linewidth=0.8)
    ax.set_title(f"Dynamic strict R2G NET: {network.network_id}")
    ax.set_ylabel("NET (%)")
    ax.set_xlabel("Window end date")
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend(ncol=2, fontsize=8, frameon=False)
    save_figure(fig, DYNAMIC_NET_DIR, f"{network.network_id}_dynamic_net_window252")



def _dynamic_net_matrix(network: NetworkDefinition, directional: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    data = directional.loc[directional["network_id"].eq(network.network_id)].copy()
    data["date"] = pd.to_datetime(data["date"])
    pivot = data.pivot_table(index="node", columns="date", values="NET", aggfunc="mean")
    pivot = pivot.reindex(index=list(network.node_columns)).sort_index(axis=1)
    return pivot, labels_for_network(network)


def _set_dynamic_date_ticks(ax: plt.Axes, dates: pd.Index) -> None:
    if len(dates) == 0:
        return
    tick_count = min(6, len(dates))
    tick_positions = np.linspace(0, len(dates) - 1, tick_count, dtype=int)
    tick_labels = [pd.Timestamp(dates[pos]).strftime("%Y") for pos in tick_positions]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=0)


def plot_dynamic_net_heatmap(network: NetworkDefinition, directional: pd.DataFrame) -> None:
    matrix, labels = _dynamic_net_matrix(network, directional)
    fig, ax = plt.subplots(figsize=(10.5, 0.42 * len(labels) + 2.0))
    values = matrix.to_numpy(dtype=float)
    vmax = max(float(np.nanmax(np.abs(values))), 1.0)
    im = ax.imshow(values, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="nearest")
    ax.set_title(f"Dynamic NET heatmap: {network.network_id}")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    _set_dynamic_date_ticks(ax, matrix.columns)
    ax.set_xlabel("Window end date")
    ax.set_ylabel("Node")
    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("NET (%)")
    save_figure(fig, DYNAMIC_NET_HEATMAP_DIR, f"{network.network_id}_dynamic_net_heatmap_window252")


def plot_dynamic_net_heatmap_group(network_ids: list[str], filename_stem: str, title: str) -> None:
    matrices = []
    labels_by_network = []
    networks_by_id = {network.network_id: network for network in networks}
    vmax = 1.0
    for network_id in network_ids:
        matrix, labels = _dynamic_net_matrix(networks_by_id[network_id], dynamic_directional_indices_window252)
        matrices.append(matrix)
        labels_by_network.append(labels)
        vmax = max(vmax, float(np.nanmax(np.abs(matrix.to_numpy(dtype=float)))))
    fig, axes = plt.subplots(len(network_ids), 1, figsize=(11.4, 2.05 * len(network_ids) + 0.9), sharex=True, squeeze=False)
    last_im = None
    for ax, network_id, matrix, labels in zip(axes.ravel(), network_ids, matrices, labels_by_network):
        last_im = ax.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax, interpolation="nearest")
        ax.set_title(_format_network_title(network_id), fontsize=12, pad=7)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=9)
    _set_dynamic_date_ticks(axes.ravel()[-1], matrices[-1].columns)
    axes.ravel()[-1].set_xlabel("Window end date")
    fig.suptitle(title, fontsize=13, y=0.98)
    fig.subplots_adjust(left=0.08, right=0.88, bottom=0.08, top=0.90, hspace=0.58)
    cax = fig.add_axes([0.905, 0.13, 0.018, 0.70])
    cbar = fig.colorbar(last_im, cax=cax)
    cbar.set_label("NET (%)")
    fig.savefig(DYNAMIC_NET_HEATMAP_DIR / f"{filename_stem}.png", bbox_inches="tight")
    fig.savefig(DYNAMIC_NET_HEATMAP_DIR / f"{filename_stem}.pdf", bbox_inches="tight")
    plt.close(fig)



for network in networks:
    plot_dynamic_net_heatmap(network, dynamic_directional_indices_window252)

plot_dynamic_net_heatmap_group(same_scale_ids, "same_scale_dynamic_net_heatmap_window252", "Dynamic NET heatmaps: same-scale networks")
plot_dynamic_net_heatmap_group(within_ids, "within_metal_dynamic_net_heatmap_window252", "Dynamic NET heatmaps: within-metal networks")

print("Dynamic TCI and NET visualizations saved.")


Dynamic TCI and NET visualizations saved.


In [10]:
combined_tci = pd.concat(
    [
        dynamic_tci_window252.assign(window_size=252),
        robustness_tci,
    ],
    ignore_index=True,
)
combined_tci["date"] = pd.to_datetime(combined_tci["date"])

for network in networks:
    data = combined_tci.loc[combined_tci["network_id"].eq(network.network_id)].copy()
    fig, ax = plt.subplots(figsize=(10.8, 5.2))
    for window_size, group in data.groupby("window_size"):
        group = group.sort_values("date")
        ax.plot(group["date"], group["TCI"], linewidth=1.1, label=f"window={window_size}")
    ax.set_title(f"TCI robustness by rolling window: {network.network_id}")
    ax.set_ylabel("TCI (%)")
    ax.set_xlabel("Window end date")
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend(frameon=False)
    save_figure(fig, ROBUSTNESS_TCI_DIR, f"{network.network_id}_tci_robustness_windows_200_252_300")

print("Robustness TCI visualizations saved.")

Robustness TCI visualizations saved.


## PCA-vs-Strict R2G Method Comparison

Static strict R2G results are compared against the notebook 04 PCA-orthogonalized explanatory connectedness outputs using identical networks and input data.

In [11]:
pca_output_dir = PROJECT_ROOT / "data" / "processed" / "04_r2_decomposed_connectedness_hierarchical_ward"

if pca_output_dir.exists():
    pca_static_tci = pd.read_csv(pca_output_dir / "static_tci.csv")
    strict_static_tci = static_tci.copy()
    method_comparison_tci = strict_static_tci.merge(
        pca_static_tci[["network_id", "TCI"]].rename(columns={"TCI": "pca_tci"}),
        on="network_id",
        how="left",
    )
    method_comparison_tci = method_comparison_tci.rename(columns={"TCI": "strict_r2g_tci"})
    method_comparison_tci["strict_minus_pca_tci"] = method_comparison_tci["strict_r2g_tci"] - method_comparison_tci["pca_tci"]
    method_comparison_tci.to_csv(OUTPUT_DIR / "method_comparison_static_tci_pca_vs_strict_r2g.csv", index=False)

    pca_directional = pd.read_csv(pca_output_dir / "static_directional_indices.csv")
    method_comparison_net = static_directional_indices.merge(
        pca_directional[["network_id", "node", "NET"]].rename(columns={"NET": "pca_NET"}),
        on=["network_id", "node"],
        how="left",
    )
    method_comparison_net = method_comparison_net.rename(columns={"NET": "strict_r2g_NET"})
    method_comparison_net["strict_minus_pca_NET"] = method_comparison_net["strict_r2g_NET"] - method_comparison_net["pca_NET"]
    method_comparison_net.to_csv(OUTPUT_DIR / "method_comparison_static_net_pca_vs_strict_r2g.csv", index=False)

    pca_npdc = pd.read_csv(pca_output_dir / "static_npdc.csv")
    method_comparison_npdc = static_npdc.merge(
        pca_npdc[["network_id", "source", "target", "NPDC"]].rename(columns={"NPDC": "pca_NPDC"}),
        on=["network_id", "source", "target"],
        how="left",
    )
    method_comparison_npdc = method_comparison_npdc.rename(columns={"NPDC": "strict_r2g_NPDC"})
    method_comparison_npdc["strict_minus_pca_NPDC"] = method_comparison_npdc["strict_r2g_NPDC"] - method_comparison_npdc["pca_NPDC"]
    method_comparison_npdc.to_csv(OUTPUT_DIR / "method_comparison_static_npdc_pca_vs_strict_r2g.csv", index=False)
else:
    method_comparison_tci = pd.DataFrame()
    method_comparison_net = pd.DataFrame()
    method_comparison_npdc = pd.DataFrame()

method_comparison_tci


,date,window_size,window_start,window_end,network_id,network_type,network_scale,network_metal,strict_r2g_tci,mean_total_r2,min_total_r2,max_total_r2,pca_tci,strict_minus_pca_tci
0,None,3100,2012-01-04,2026-03-31,same_scale_STC,same_scale_cross_metal,STC,NaN,79.510360,0.795104,0.622962,0.885223,79.510360,2.842171e-14
1,None,3100,2012-01-04,2026-03-31,same_scale_MTC,same_scale_cross_metal,MTC,NaN,65.334276,0.653343,0.115359,0.877486,65.334276,-2.842171e-14
2,None,3100,2012-01-04,2026-03-31,same_scale_LTC,same_scale_cross_metal,LTC,NaN,70.787508,0.707875,0.591603,0.823226,70.787508,0.000000e+00
3,None,3100,2012-01-04,2026-03-31,within_gold,within_metal_cross_scale,NaN,gold,57.576710,0.575767,0.427603,0.703990,57.576710,0.000000e+00
4,None,3100,2012-01-04,2026-03-31,within_silver,within_metal_cross_scale,NaN,silver,49.404760,0.494048,0.195397,0.750371,49.404760,-1.421085e-14
5,None,3100,2012-01-04,2026-03-31,within_platinum,within_metal_cross_scale,NaN,platinum,84.686290,0.846863,0.798309,0.891780,84.686290,1.421085e-14
6,None,3100,2012-01-04,2026-03-31,within_palladium,within_metal_cross_scale,NaN,palladium,77.467253,0.774673,0.582302,0.881196,77.467253,1.421085e-14


## Manifest and Final Checks

The final checks verify row counts, reconstruction network coverage, numerical identities, and expected output files.

In [12]:
def relpath(path: Path) -> str:
    return str(path.relative_to(PROJECT_ROOT)).replace("\\", "/")


expected_rows = {
    "dynamic_tci_window252": (len(components_wide) - 252 + 1) * len(networks),
    "dynamic_tci_robustness_window200": (len(components_wide) - 200 + 1) * len(networks),
    "dynamic_tci_robustness_window300": (len(components_wide) - 300 + 1) * len(networks),
}

actual_rows = {
    "dynamic_tci_window252": int(len(dynamic_tci_window252)),
    "dynamic_tci_robustness_window200": int((robustness_tci["window_size"] == 200).sum()),
    "dynamic_tci_robustness_window300": int((robustness_tci["window_size"] == 300).sum()),
}
assert actual_rows == expected_rows, {"actual": actual_rows, "expected": expected_rows}

max_validation_error = max(
    float(static_validation["row_error"].max()),
    float(dynamic_validation_window252["max_row_error"].max()),
    float(robustness_validation["max_row_error"].max()),
)
assert max_validation_error < PARAMS["r2_tolerance"]

required_outputs = [
    OUTPUT_DIR / "network_node_metadata.csv",
    OUTPUT_DIR / "static_r2g_matrices_long.csv",
    OUTPUT_DIR / "static_directional_indices.csv",
    OUTPUT_DIR / "static_npdc.csv",
    OUTPUT_DIR / "static_tci.csv",
    OUTPUT_DIR / "dynamic_tci_window252.csv",
    OUTPUT_DIR / "dynamic_directional_indices_window252.csv",
    OUTPUT_DIR / "dynamic_npdc_window252.csv",
    OUTPUT_DIR / "dynamic_tci_robustness_windows_200_300.csv",
]
for output_file in required_outputs:
    assert output_file.exists(), output_file

figure_counts = {
    "static_heatmaps_png": len(list(STATIC_HEATMAP_DIR.glob("*.png"))),
    "static_heatmaps_pdf": len(list(STATIC_HEATMAP_DIR.glob("*.pdf"))),
    "static_net_png": len(list(STATIC_NET_DIR.glob("*.png"))),
    "static_net_pdf": len(list(STATIC_NET_DIR.glob("*.pdf"))),
    "static_network_png": len(list(STATIC_NETWORK_DIR.glob("*.png"))),
    "static_network_pdf": len(list(STATIC_NETWORK_DIR.glob("*.pdf"))),
    "dynamic_tci_png": len(list(DYNAMIC_TCI_DIR.glob("*.png"))),
    "dynamic_tci_pdf": len(list(DYNAMIC_TCI_DIR.glob("*.pdf"))),
    "dynamic_net_png": len(list(DYNAMIC_NET_DIR.glob("*.png"))),
    "dynamic_net_pdf": len(list(DYNAMIC_NET_DIR.glob("*.pdf"))),
    "dynamic_net_heatmap_png": len(list(DYNAMIC_NET_HEATMAP_DIR.glob("*.png"))),
    "dynamic_net_heatmap_pdf": len(list(DYNAMIC_NET_HEATMAP_DIR.glob("*.pdf"))),
    "robustness_tci_png": len(list(ROBUSTNESS_TCI_DIR.glob("*.png"))),
    "robustness_tci_pdf": len(list(ROBUSTNESS_TCI_DIR.glob("*.pdf"))),
}

assert figure_counts["static_heatmaps_png"] == len(networks)
assert figure_counts["static_net_png"] == len(networks)
assert figure_counts["static_network_png"] == len(networks)
assert figure_counts["dynamic_tci_png"] == 3
assert figure_counts["dynamic_net_png"] == len(networks)
assert figure_counts["dynamic_net_heatmap_png"] == len(networks) + 2
assert figure_counts["robustness_tci_png"] == len(networks)

manifest = {
    "method": "Strict paper-formula R2G decomposed connectedness on Hierarchical Ward STC/MTC/LTC reconstructed components.",
    "method_note": "This notebook implements the strict paper-formula R2G decomposition. r2g[target, source] columns store source-node R2G contribution shares to the target-node multiple-regression R2. FROM is row sum, TO is column sum, NET is TO minus FROM, and TCI is the mean row sum. The decomposition is contemporaneous; causal timing is assessed separately in notebook 06.",
    "parameters": PARAMS,
    "input_checks": input_checks,
    "source_reconstruction_summary": {
        "source_method": source_manifest.get("method"),
        "source_parameters": source_manifest.get("parameters"),
        "source_date_start": source_manifest.get("input_checks", {}).get("date_start"),
        "source_date_end": source_manifest.get("input_checks", {}).get("date_end"),
    },
    "networks": [
        {
            "network_id": network.network_id,
            "network_type": network.network_type,
            "scale": network.scale,
            "metal": network.metal,
            "node_columns": list(network.node_columns),
        }
        for network in networks
    ],
    "validation": {
        "max_r2_row_error": max_validation_error,
        "expected_rows": expected_rows,
        "actual_rows": actual_rows,
        "figure_counts": figure_counts,
    },
    "outputs": {
        "network_node_metadata": relpath(OUTPUT_DIR / "network_node_metadata.csv"),
        "static_r2g_matrices_long": relpath(OUTPUT_DIR / "static_r2g_matrices_long.csv"),
        "static_directional_indices": relpath(OUTPUT_DIR / "static_directional_indices.csv"),
        "static_npdc": relpath(OUTPUT_DIR / "static_npdc.csv"),
        "static_tci": relpath(OUTPUT_DIR / "static_tci.csv"),
        "static_r2_decomposition_validation": relpath(OUTPUT_DIR / "static_r2_decomposition_validation.csv"),
        "dynamic_tci_window252": relpath(OUTPUT_DIR / "dynamic_tci_window252.csv"),
        "dynamic_directional_indices_window252": relpath(OUTPUT_DIR / "dynamic_directional_indices_window252.csv"),
        "dynamic_npdc_window252": relpath(OUTPUT_DIR / "dynamic_npdc_window252.csv"),
        "dynamic_r2_decomposition_validation_window252": relpath(OUTPUT_DIR / "dynamic_r2_decomposition_validation_window252.csv"),
        "dynamic_tci_robustness_windows_200_300": relpath(OUTPUT_DIR / "dynamic_tci_robustness_windows_200_300.csv"),
        "method_comparison_static_tci_pca_vs_strict_r2g": relpath(OUTPUT_DIR / "method_comparison_static_tci_pca_vs_strict_r2g.csv"),
        "method_comparison_static_net_pca_vs_strict_r2g": relpath(OUTPUT_DIR / "method_comparison_static_net_pca_vs_strict_r2g.csv"),
        "method_comparison_static_npdc_pca_vs_strict_r2g": relpath(OUTPUT_DIR / "method_comparison_static_npdc_pca_vs_strict_r2g.csv"),
        "dynamic_r2_decomposition_validation_robustness_windows_200_300": relpath(OUTPUT_DIR / "dynamic_r2_decomposition_validation_robustness_windows_200_300.csv"),
        "visualizations": relpath(VIS_DIR),
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "networkx": nx.__version__,
    },
    "runtime_seconds": float(time.perf_counter() - start_time),
}

manifest_path = OUTPUT_DIR / "strict_r2g_decomposed_connectedness_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(manifest["validation"], ensure_ascii=False, indent=2))
print(f"Manifest saved to {relpath(manifest_path)}")


{
  "max_r2_row_error": 5.551115123125783e-16,
  "expected_rows": {
    "dynamic_tci_window252": 19943,
    "dynamic_tci_robustness_window200": 20307,
    "dynamic_tci_robustness_window300": 19607
  },
  "actual_rows": {
    "dynamic_tci_window252": 19943,
    "dynamic_tci_robustness_window200": 20307,
    "dynamic_tci_robustness_window300": 19607
  },
  "figure_counts": {
    "static_heatmaps_png": 7,
    "static_heatmaps_pdf": 7,
    "static_net_png": 7,
    "static_net_pdf": 7,
    "static_network_png": 7,
    "static_network_pdf": 7,
    "dynamic_tci_png": 3,
    "dynamic_tci_pdf": 3,
    "dynamic_net_png": 7,
    "dynamic_net_pdf": 7,
    "robustness_tci_png": 7,
    "robustness_tci_pdf": 7
  }
}


Manifest saved to data/processed/05_strict_r2g_decomposed_connectedness_hierarchical_ward/strict_r2g_decomposed_connectedness_manifest.json
